# Week 04: EDA Asks What the Data Can Support

This notebook follows the reviewed Week 4 presentation. Read each concept and calculate the worked example before running its code.

## Lesson map

1. EDA Asks What the Data Can Support
2. Start with Schema, Grain, and Quality Rules
3. Center Describes a Typical Value
4. Spread Describes How Values Vary
5. A Distribution Shows Frequency and Shape
6. Outliers Are Signals to Investigate
7. Missing Markers and Categories Need Counting
8. Class Imbalance Changes How Accuracy Feels
9. Relationships Do Not Automatically Mean Causation
10. Feature Engineering Creates Measurable Inputs
11. A Reproducible EDA Has a Question and Evidence
12. Guided Lab: Produce Three Defensible Findings

Use the same reasoning loop throughout: **predict, run, inspect, explain**.


## 1. EDA Asks What the Data Can Support

**Exploratory data analysis (EDA)** is the process of examining a dataset to understand its structure, quality, distributions, and relationships.

EDA helps answer:

- What does one row represent?
- Which values are missing, invalid, or unusual?
- What values are typical?
- How spread out are the values?
- Which groups differ?
- Which relationships deserve further investigation?

EDA produces evidence and new questions. It does not automatically prove why a pattern exists.

### Work it out first

Question: “Do customers who subscribed differ in age from those who did not?”

Before comparing ages, check the row grain, missing values, valid age range, and the number of customers in each group.

### Notebook bridge

The EDA notebook begins with data loading and an initial assessment before feature work.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
print(df.shape)
print(df["subscribed"].value_counts(dropna=False))

Expected output:

```text
One row count and the number of records in each target class.
```


## 2. Start with Schema, Grain, and Quality Rules

A **schema** describes column names, data types, and expected meanings. A **quality rule** states what a valid value or row must satisfy.

Examples:

- age must be between `18` and `100`;
- customer ID must be unique;
- balance may be negative, so “negative” is not automatically invalid;
- `"unknown"` may be a missing-value marker rather than a real category.

Data types describe storage. They do not guarantee meaning. A numeric age column can still contain impossible values.

### Work it out first

If a table has `4,521` rows but only `4,510` unique customer IDs, at least `11` rows repeat an ID. The duplicates must be investigated before treating rows as independent customers.

### Notebook bridge

This prepares learners for the notebook's initial assessment and data-quality checks.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
print(df.dtypes)
print(df["customer_id"].nunique())
print(df["age"].between(18, 100).all())

Expected output:

```text
Column types, a unique-ID count, and True or False for the age rule.
```


## 3. Center Describes a Typical Value

The **mean** is the sum divided by the number of values:

`mean = (x1 + x2 + ... + xn) / n`

The **median** is the middle value after sorting.

The mean uses every value and is pulled toward extremes. The median depends on order and is more resistant to extreme values. Neither is always “better”; choose the statistic that answers the question and describe the distribution.

### Work it out first

Values: `[20, 22, 24, 26, 200]`

Mean:

`(20 + 22 + 24 + 26 + 200) / 5 = 292 / 5 = 58.4`

Median: the ordered middle value is `24`.

The value `200` pulls the mean far above most observations.

### Notebook bridge

Learners will compare descriptive statistics and explain why one summary can hide skew.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
values = pd.Series([20, 22, 24, 26, 200])
print(values.mean(), values.median())

Expected output:

```text
58.4 24.0
```


## 4. Spread Describes How Values Vary

**Spread** describes how far values vary.

- **Range:** maximum minus minimum
- **Variance:** average squared distance from the mean
- **Standard deviation:** square root of variance
- **Interquartile range (IQR):** third quartile minus first quartile

Standard deviation uses every value and has the same units as the data. IQR describes the middle half of ordered values and is less affected by extremes.

### Work it out first

For `[2, 4, 6]`, mean is `4`.

Squared distances: `(2-4)^2 = 4`, `(4-4)^2 = 0`, `(6-4)^2 = 4`

Population variance: `(4 + 0 + 4) / 3 = 2.67`  
Population standard deviation: `sqrt(2.67) ≈ 1.63`

### Notebook bridge

Spread helps learners compare variables and detect distributions that need closer inspection.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
values = pd.Series([2, 4, 6])
print(values.var(ddof=0))
print(values.std(ddof=0))

Expected output:

```text
2.6666666666666665
1.632993161855452
```


## 5. A Distribution Shows Frequency and Shape

A **distribution** describes which values occur and how often.

A **histogram** divides a numerical range into intervals called bins. Bar height shows how many observations fall inside each interval.

Common shapes include:

- roughly symmetric;
- right-skewed with a long high-value tail;
- left-skewed with a long low-value tail;
- multimodal with more than one peak.

Changing bin width can reveal or hide structure, so inspect more than one reasonable setting.

### Work it out first

Scores `[42, 48, 51, 55, 56, 82, 88]` have a cluster around 42–56 and a smaller cluster around 82–88. The mean alone would not show both groups.

### Notebook bridge

The notebook's visualizations should be interpreted in words, not included as decoration.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
df["score"].plot.hist(bins=5)

Expected output:

```text
A histogram with score intervals on the x-axis and counts on the y-axis.
```


## 6. Outliers Are Signals to Investigate

An **outlier** is an observation unusually far from most values under a stated rule.

One common flag uses the IQR:

- lower fence: `Q1 - 1.5 x IQR`
- upper fence: `Q3 + 1.5 x IQR`

A value beyond a fence is a candidate for investigation. It may be a data error, a rare valid case, or evidence of a separate group. Removing it without checking can erase important information.

### Work it out first

If `Q1 = 20` and `Q3 = 40`:

`IQR = 40 - 20 = 20`  
Lower fence: `20 - 1.5 x 20 = -10`  
Upper fence: `40 + 1.5 x 20 = 70`

A value of `95` is flagged because `95 > 70`.

### Notebook bridge

Learners should record how unusual values are detected and what decision follows.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
q1 = df["value"].quantile(0.25)
q3 = df["value"].quantile(0.75)
iqr = q3 - q1
outliers = df[~df["value"].between(q1 - 1.5*iqr, q3 + 1.5*iqr)]

Expected output:

```text
A DataFrame containing values outside the stated fences.
```


## 7. Missing Markers and Categories Need Counting

Missing information may appear as `NaN`, an empty string, `-1`, `"unknown"`, or `"not applicable"`. These are **sentinel values** when they stand in for another meaning.

For categorical columns:

- count each category;
- include missing values;
- inspect spelling and capitalization;
- confirm whether sentinel values are real categories;
- report both counts and percentages.

A rare category may be valid and important. Combining categories requires a stated reason.

### Work it out first

Job values:

```text
teacher, teacher, unknown, Teacher, missing
```

Without cleaning, `teacher` and `Teacher` are counted separately. `"unknown"` and missing may represent two different data-collection outcomes.

### Notebook bridge

This prepares the notebook's quality checks and categorical visualizations.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
print(df["job"].value_counts(dropna=False))

Expected output:

```text
One count for every observed category, including missing values.
```


## 8. Class Imbalance Changes How Accuracy Feels

A **target** is the outcome a supervised model will learn to predict. For a categorical target, each possible value is a **class**.

**Class imbalance** occurs when one class is much more common than another.

If `90` of `100` customers did not subscribe, a rule that always predicts “no” is `90%` accurate. It still identifies none of the `10` subscribers.

This is why EDA must count target classes before modelling. The appropriate evaluation measure depends on which mistakes matter.

### Work it out first

Actual counts:

- No: `90`
- Yes: `10`

Always predict No:

`accuracy = 90 correct / 100 total = 0.90`

Subscribers found: `0 / 10 = 0`.

### Notebook bridge

The notebook's target distribution should be described before feature importance or modelling.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
counts = df["subscribed"].value_counts()
percent = df["subscribed"].value_counts(normalize=True)
print(counts, percent)

Expected output:

```text
Counts and proportions for each class.
```


## 9. Relationships Do Not Automatically Mean Causation

An **association** means two variables vary together in the observed data.

**Correlation** summarizes the direction and strength of a linear relationship between two numerical variables. Pearson correlation ranges from `-1` to `1`.

- near `1`: strong positive linear relationship;
- near `-1`: strong negative linear relationship;
- near `0`: weak linear relationship.

**Causation** means changing one factor produces a change in another. Correlation alone does not establish causation because confounding factors, selection, timing, or chance may explain the pattern.

### Work it out first

Suppose study hours and scores have correlation `0.70`. This supports a positive association in this dataset. It does not prove that adding one study hour causes a fixed score increase.

### Notebook bridge

Learners should use cautious language when interpreting correlations and feature importance.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
print(df[["study_hours", "score"]].corr())

Expected output:

```text
A 2-by-2 correlation matrix with 1.0 on the diagonal.
```


## 10. Feature Engineering Creates Measurable Inputs

A **feature** is an input variable available when a prediction must be made. **Feature engineering** creates a feature from existing data using a stated rule.

Examples:

- `contact_duration_minutes = seconds / 60`
- `has_previous_contact = previous_contacts > 0`
- extract month from a valid contact date

A useful feature has:

- a clear meaning;
- a reproducible formula;
- valid source columns;
- no future information;
- checks for missing or impossible results.

### Work it out first

If contact duration is `150` seconds:

`150 / 60 = 2.5` minutes

If duration is missing, the engineered value is also missing unless a documented policy says otherwise.

### Notebook bridge

The notebook creates features after initial assessment; learners must explain each transformation.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
df["duration_minutes"] = df["duration_seconds"] / 60
print(df[["duration_seconds", "duration_minutes"]].head())

Expected output:

```text
Each valid seconds value paired with its value divided by 60.
```


## 11. A Reproducible EDA Has a Question and Evidence

For each EDA question:

1. state the question;
2. identify required columns and row grain;
3. check missing, invalid, and duplicate values;
4. calculate a relevant statistic;
5. create a chart that answers the question;
6. interpret the pattern in plain language;
7. record limitations and the next question.

Every chart needs a title, labelled axes, units, and a written interpretation. Every transformation needs an inspectable output.

### Work it out first

Question: “How does account balance differ by subscription outcome?”

Evidence:

- group counts;
- median and IQR of balance per class;
- side-by-side box plots;
- note that the comparison is observational and may be affected by other variables.

### Notebook bridge

This workflow maps directly to the notebook's assessment, statistics, visualization, and feature sections.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
summary = df.groupby("subscribed")["balance"].agg(
    count="count", median="median"
)
print(summary)

Expected output:

```text
One row per class with its non-missing count and median balance.
```


## 12. Guided Lab: Produce Three Defensible Findings

Use the Bank Marketing dataset to produce three findings.

Your report must include:

1. row grain and target definition;
2. shape, types, missing markers, and duplicate checks;
3. target class counts and percentages;
4. one numerical distribution with center and spread;
5. one group comparison;
6. one relationship between numerical variables;
7. one documented feature idea;
8. limitations and an unanswered question.

Do not make causal claims from correlation or feature importance.

### Work it out first

Finding format:

“The median balance was higher in group A than group B. Group sizes were `nA` and `nB`. This is an observed association; the data does not establish that balance caused the outcome.”

### Notebook bridge

Continue into `02.exploratory-data-analysis.ipynb`, treating notebook cells as the lab rather than the lesson outline.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
assert len(df) > 0
assert df["subscribed"].notna().all()

Expected output:

```text
Both assertions pass before target analysis begins.
```


## Guided lab

Use the Bank Marketing dataset to produce three findings.

Your report must include:

1. row grain and target definition;
2. shape, types, missing markers, and duplicate checks;
3. target class counts and percentages;
4. one numerical distribution with center and spread;
5. one group comparison;
6. one relationship between numerical variables;
7. one documented feature idea;
8. limitations and an unanswered question.

Do not make causal claims from correlation or feature importance.

### Reference result

Finding format:

“The median balance was higher in group A than group B. Group sizes were `nA` and `nB`. This is an observed association; the data does not establish that balance caused the outcome.”


In [ ]:
# Guided lab workspace: Week 04
# Add only the imports needed for the current step.

# TODO 1: Prepare the smallest valid input.

# TODO 2: Apply the concept taught in this lesson.

# TODO 3: Display inspectable intermediate evidence.

# TODO 4: Compare the result with a hand calculation or stated requirement.

## Weekly deliverable

Submit the completed guided lab with:

- your prediction before execution;
- intermediate values, shapes, metrics, or traces;
- one failed assumption and its correction;
- a plain-English explanation of the result;
- the source notebook section you are now ready to complete.


## Sources and source notebooks

- <https://github.com/curiousily/AI-Bootcamp/blob/master/02.exploratory-data-analysis.ipynb>
- <https://www.itl.nist.gov/div898/handbook/eda/eda.htm>
- <https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dtypes.html>
- <https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.nunique.html>
- <https://www.itl.nist.gov/div898/handbook/eda/section3/eda351.htm>
- <https://pandas.pydata.org/docs/reference/api/pandas.Series.describe.html>
- <https://www.itl.nist.gov/div898/handbook/eda/section3/eda356.htm>
- <https://pandas.pydata.org/docs/reference/api/pandas.Series.var.html>
- <https://www.itl.nist.gov/div898/handbook/eda/section3/histogra.htm>
- <https://pandas.pydata.org/docs/user_guide/visualization.html>
- <https://www.itl.nist.gov/div898/handbook/eda/section3/boxplot.htm>
- <https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.quantile.html>
- <https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html>
- <https://pandas.pydata.org/docs/user_guide/missing_data.html>
- <https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics>
- <https://www.itl.nist.gov/div898/handbook/eda/section3/scatterp.htm>
- <https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html>
- <https://scikit-learn.org/stable/common_pitfalls.html>
- <https://www.itl.nist.gov/div898/handbook/eda/section1/eda11.htm>
- <https://archive.ics.uci.edu/dataset/222/bank+marketing>